# Topic 6: Scikit-learn Pipelines for Production
**Module 1 - Introduction to Machine Learning in Python**


In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, uniform
import joblib


## 1. Create Mixed-Type Dataset


In [ ]:
np.random.seed(42)
n = 10000
df = pd.DataFrame({
    'income': np.where(np.random.random(n)>0.08, np.random.lognormal(11,0.7,n), np.nan),
    'loan_amnt': np.random.lognormal(9.5, 0.5, n).round(0),
    'dti': np.random.uniform(0, 45, n).round(2),
    'credit_score': np.where(np.random.random(n)>0.05, np.random.normal(700,50,n).clip(300,850), np.nan),
    'grade': np.random.choice(['A','B','C','D','E','F','G'], n, p=[.15,.25,.25,.15,.10,.06,.04]),
    'purpose': np.random.choice(['debt_consolidation','credit_card','home_improvement','other'], n),
    'home_ownership': np.random.choice(['RENT','MORTGAGE','OWN'], n, p=[.42,.43,.15]),
})
grade_prob = {'A':.03,'B':.05,'C':.08,'D':.12,'E':.18,'F':.25,'G':.35}
df['default'] = df['grade'].map(grade_prob).apply(lambda p: np.random.binomial(1,p))

y = df['default']
X = df.drop('default', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


## 2. Build a Complete Pipeline


In [ ]:
numeric_features = ['income', 'loan_amnt', 'dti', 'credit_score']
categorical_features = ['grade', 'purpose', 'home_ownership']

# Numeric: impute missing with median, then scale
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# Categorical: impute missing with most frequent, then one-hot encode
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# Combine
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

# Full pipeline with model
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)),
])

print('Pipeline steps:')
print(pipeline)


## 3. Evaluate Pipeline with Cross-Validation


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(pipeline, X_train, y_train, cv=cv,
                       scoring=['roc_auc', 'average_precision'])

print(f'AUROC: {scores["test_roc_auc"].mean():.4f} (+/- {scores["test_roc_auc"].std():.4f})')
print(f'AUPRC: {scores["test_average_precision"].mean():.4f} (+/- {scores["test_average_precision"].std():.4f})')


## 4. Compare Models Inside the Pipeline


In [ ]:
# Swap classifier inside the same pipeline
for name, clf in [('Logistic Regression', LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced')),
                   ('Random Forest', RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42))]:
    pipeline.set_params(classifier=clf)
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring='roc_auc')
    print(f'{name:>25}: AUROC = {scores["test_score"].mean():.4f} (+/- {scores["test_score"].std():.4f})')


## 5. Hyperparameter Tuning Within Pipeline


In [ ]:
# Reset to Random Forest
pipeline.set_params(classifier=RandomForestClassifier(class_weight='balanced', random_state=42))

# Note: parameter names use double underscore convention
param_dist = {
    'classifier__n_estimators': randint(100, 500),
    'classifier__max_depth': [5, 10, 15, 20, None],
    'classifier__min_samples_leaf': randint(5, 50),
}

search = RandomizedSearchCV(
    pipeline, param_dist, n_iter=20, cv=3, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=0
)
search.fit(X_train, y_train)

print(f'Best CV AUROC: {search.best_score_:.4f}')
print(f'Best params: {search.best_params_}')


## 6. Final Evaluation and Save


In [ ]:
# Evaluate best pipeline on test set
best_pipeline = search.best_estimator_
y_proba = best_pipeline.predict_proba(X_test)[:, 1]
test_auroc = roc_auc_score(y_test, y_proba)
print(f'Test AUROC: {test_auroc:.4f}')

# Save
joblib.dump(best_pipeline, 'credit_scoring_pipeline.pkl')
print('Pipeline saved to credit_scoring_pipeline.pkl')

# Verify: load and predict
loaded = joblib.load('credit_scoring_pipeline.pkl')
verify_proba = loaded.predict_proba(X_test)[:, 1]
assert np.allclose(y_proba, verify_proba), 'Predictions do not match!'
print('Verification passed: loaded pipeline produces identical predictions')


## 7. Test with New Raw Data


In [ ]:
# Simulate a new applicant (raw data, no preprocessing)
new_applicant = pd.DataFrame([{
    'income': 65000,
    'loan_amnt': 15000,
    'dti': 22.5,
    'credit_score': 720,
    'grade': 'B',
    'purpose': 'debt_consolidation',
    'home_ownership': 'MORTGAGE',
}])

prediction = loaded.predict_proba(new_applicant)[:, 1][0]
print(f'New applicant predicted default probability: {prediction:.4f}')
print(f'Decision: {"APPROVE" if prediction < 0.15 else "REVIEW" if prediction < 0.30 else "DECLINE"}')
